# Gold — `gold_alertas_qualidade`

## Bibliotecas

In [0]:
from pyspark.sql import functions as F

## Variáveis

In [0]:
# origem: as 5 quarentenas da silver
quarentena_clientes   = "novarota.silver.quarentena_clientes"
quarentena_contas     = "novarota.silver.quarentena_contas"
quarentena_cartoes    = "novarota.silver.quarentena_cartoes"
quarentena_transacoes = "novarota.silver.quarentena_transacoes"
quarentena_estornos   = "novarota.silver.quarentena_estornos"
silver_controle       = "novarota.silver.controle_cargas"

# destino
tabela_gold     = "novarota.gold.gold_alertas_qualidade"
tabela_controle = "novarota.gold.controle_gold"
nome_tabela     = "gold_alertas_qualidade"

# chave da tabela: um motivo de erro por tabela por carga
chaves = ["tabela", "motivo_erro", "data_referencia"]

print(f"Tabela alvo: {tabela_gold}")

## Ultima carga

In [0]:
# cargas que a silver terminou. Uso o estornos porque é o último notebook da silver: quando ele registra,
# as 5 quarentenas daquela carga já foram gravadas
cargas_silver = [
    l.data_referencia
    for l in spark.table(silver_controle).filter("tabela = 'estornos'").select("data_referencia").distinct().collect()
]

# cargas que essa tabela da gold já processou
if spark.catalog.tableExists(tabela_controle):
    cargas_processadas = [
        l.data_referencia
        for l in spark.table(tabela_controle).filter(f"tabela = '{nome_tabela}'").select("data_referencia").distinct().collect()
    ]
else:
    cargas_processadas = []

cargas_pendentes = sorted(c for c in cargas_silver if c not in cargas_processadas)
cargas_sql = ", ".join(f"'{c}'" for c in cargas_pendentes)

print(f"Cargas já processadas: {sorted(cargas_processadas)}")
print(f"Cargas pendentes: {cargas_pendentes}")

if not cargas_pendentes:
    dbutils.notebook.exit(f"Nada pendente pra {nome_tabela}")

## Transformações

In [0]:
query = f"""
with quarentena as (
select 'clientes'   as tabela, motivo_erro, data_referencia from {quarentena_clientes}
union all
select 'contas'     as tabela, motivo_erro, data_referencia from {quarentena_contas}
union all
select 'cartoes'    as tabela, motivo_erro, data_referencia from {quarentena_cartoes}
union all
select 'transacoes' as tabela, motivo_erro, data_referencia from {quarentena_transacoes}
union all
select 'estornos'   as tabela, motivo_erro, data_referencia from {quarentena_estornos})


select
    tabela,
    motivo_erro,
    data_referencia,
    count(*) as qtd_registros
from quarentena
where data_referencia in ({cargas_sql})
group by tabela, motivo_erro, data_referencia
"""

df = spark.sql(query)

print(f"Linhas de alerta das cargas pendentes: {df.count()}")

## Conferência

In [0]:
dups_no_lote = (
    df.groupBy(*chaves).count()
    .filter(F.col("count") > 1)
    .count()
)
print(f"Alertas duplicados no lote (esperado 0): {dups_no_lote}")

df.orderBy("data_referencia", "tabela", "motivo_erro").display()

## Gravação por carga

In [0]:
if not spark.catalog.tableExists(tabela_gold):
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(tabela_gold)
    )
    print(f"Tabela criada (carga inicial): {tabela_gold} | {df.count()} linhas")
else:
    # aqui o recorte é a carga, não o mês: substitui só as cargas pendentes
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("replaceWhere", f"data_referencia in ({cargas_sql})")
        .saveAsTable(tabela_gold)
    )
    print(f"Cargas regravadas em {tabela_gold}: {cargas_pendentes}")

## Controle

In [0]:
# registra as cargas que essa tabela processou. Fica por último: só conta se a gravação deu certo.
# o delete antes evita linha repetida no log se a célula rodar de novo
controle = (
    spark.createDataFrame([(nome_tabela, c) for c in cargas_pendentes], "tabela string, data_referencia string")
    .withColumn("processado_em", F.current_timestamp())
)
if spark.catalog.tableExists(tabela_controle):
    spark.sql(f"delete from {tabela_controle} where tabela = '{nome_tabela}' and data_referencia in ({cargas_sql})")
controle.write.format("delta").mode("append").saveAsTable(tabela_controle)

controle.display()

## Conferência pós-escrita

In [0]:
print(f"Conferência da tabela: {tabela_gold}")

df_gold = spark.table(tabela_gold)
print(f"Total de linhas: {df_gold.count()}")

dups = (
    df_gold.groupBy(*chaves).count()
    .filter(F.col("count") > 1)
    .count()
)
print(f"Alertas duplicados (esperado 0): {dups}")

df_gold.orderBy("data_referencia", "tabela", "motivo_erro").display()